# Create the Student Direct Lake Model

Run this setup before class in your own Fabric workspace with a populated Northstar Warehouse and Semantic Link available. No Lakehouse attachment or separate model upload is required.

Set the three names in Cell 2, then select Run all. A blank workspace name uses this notebook's workspace. The model name must be unused. This notebook creates and verifies a new three-table Direct Lake on OneLake model; it never updates Warehouse rows or replaces existing models.

After setup, disable Keep your Direct Lake data up to date on the new model in Fabric settings. Use the supplied SQL scripts only in a Warehouse you own, with no other writers during the exercise.


In [ ]:
workspace_name = ""
warehouse_name = "wh_northstar"
semantic_model_name = "sm_northstar_connectivity_demo"


In [ ]:
import base64
import hashlib
import json
import re
import time
import zlib
from urllib.parse import quote, urlparse
from uuid import UUID

try:
    import notebookutils
    import sempy.fabric as fabric
except ImportError as error:
    raise RuntimeError("Run this notebook in Fabric with Semantic Link available, not a local Python kernel.") from error

fabric_client = fabric.FabricRestClient(retry_config={"total": 0})
power_bi_client = fabric.PowerBIRestClient(retry_config={"total": 0})
operation_timeout_seconds = 600
model_id = None
model_created = False
refresh_completed = False
persisted_definition_validated = False


def require_success(response, action, allowed_statuses=(200,)):
    if response.status_code not in allowed_statuses:
        try:
            details = json.dumps(response.json(), ensure_ascii=True)
        except ValueError:
            details = response.text
        raise RuntimeError(f"{action} failed (HTTP {response.status_code}): {details[:1500]}")
    return response


def list_all(path):
    values = []
    seen_tokens = set()
    base_path = path
    while path:
        page = require_success(fabric_client.get(path, timeout=120), "List Fabric items").json()
        values.extend(page.get("value", []))
        token = page.get("continuationToken")
        if token:
            if token in seen_tokens:
                raise RuntimeError("Fabric returned a repeated continuation token. Listing stopped.")
            seen_tokens.add(token)
            separator = "&" if "?" in base_path else "?"
            path = f"{base_path}{separator}continuationToken={quote(token, safe='')}"
        else:
            path = None
    return values


def unique_name(items, name, label):
    matches = [item for item in items if item["displayName"].casefold() == name.casefold()]
    if len(matches) != 1:
        raise ValueError(f"Expected one {label} named {name!r}; found {len(matches)}. Check the name and your access.")
    return matches[0]


def wait_for_operation(response, action):
    require_success(response, action, (200, 201, 202))
    if response.status_code != 202:
        return response.json()
    operation_id = response.headers.get("x-ms-operation-id") or response.headers.get("Operation-Id")
    if not operation_id:
        match = re.search(r"/operations/([0-9a-fA-F-]+)", response.headers.get("Location", ""))
        operation_id = match.group(1) if match else None
    if not operation_id:
        raise RuntimeError(f"{action} was accepted without a tracking ID. Inspect the workspace; do not resubmit blindly.")
    operation_id = str(UUID(operation_id))
    print(f"{action} operation: {operation_id}")
    deadline = time.monotonic() + operation_timeout_seconds
    previous_status = None
    while time.monotonic() < deadline:
        state_response = require_success(fabric_client.get(f"/v1/operations/{operation_id}", timeout=120), f"Check {action}")
        state = state_response.json()
        status = state.get("status")
        if status != previous_status:
            print(f"{action}: {status}")
            previous_status = status
        if status == "Succeeded":
            return require_success(fabric_client.get(f"/v1/operations/{operation_id}/result", timeout=120), f"Get {action} result").json()
        if status in {"Failed", "Cancelled", "Canceled"}:
            raise RuntimeError(f"{action}: {json.dumps(state, ensure_ascii=True)}")
        delay = float(state_response.headers.get("Retry-After", 10))
        time.sleep(min(max(delay, 1), 30, max(0, deadline - time.monotonic())))
    raise TimeoutError(f"{action} remains pending. Operation ID: {operation_id}. Inspect it before another run.")


print("Libraries loaded. Requests are not automatically retried; existing models are never updated or deleted.")


In [ ]:
template_sha256 = "89d62d259ab68bf86bf6458d4f68a49d71562aa6fdd200f696baab859c439bcc"
template_bundle_zlib_base64 = "eNq9WGtvG7cS/SuEU8AtYEr7fqjIh8ROEaOxnVpxgiAqDC53aLPdh0py46hB/vsd7q6k1VrxjRPc6w+GtORy5pyZMzPU5wMhC9AHs88HOQhZSSPrarLMpC4PZgefFxUhi4OfNL+Fki0OZvjl1pilnk2nOXyEol6CmpSSq1rXwkx4XU7/0nVFuxf0VLBMST6VBsqpxieVkfyszqGYbq29VvYUI0FP3YkzcabdyxN70OLgqHPhIygt7XfrQjDxNgsajJHVjbYrrbv24T8Ve1GxrIDcPhas0GCXviyqLwdHA6DTnBmWMQ0TU+YFAp5Op2RumhwqQ06kAm7IK/Y3kLoiFxW0HxHjkimJzpHSIiHshslKG2Jugeju3UNN6ruKnNfK3GrDFHnHFNzWDRpaVGubRJfX1XrHtVCsRBzXOZT1olqY1oyRmSykWb2yXM+IGzserlW7GODTUoG27OghjK+6X1X4HHcTU4983nqJrpP5H68IVPmylkhGrfoD/2hArSylRcb43whna54cdltaO3SA/ri3+RGhHJKnFt7CFGC6DzZk87pRHMhT8uzfRsHc1IrdwOQEmbKH/dw73+26UsUR+fBSgmKK30rOinP2Ud4wi+mpUQ38+Ut3sKzGBuz3QlaAh79hNzMSJSGHMA8o4wmjAWQhTbLApxCHHosjHoRhYAm3fK4JVHWNdIg94V7Wy6ZgBvK9gSfP66bKSbbq3gTTLElVG8jqekTjGC3SspVdXUGBi5Nc6EmnrpH+3l9cXV6/u7j8ff762fGL69OT/smzyxcvL67m9snigJRgGPlwql8zzDswoNq4tvQdkTerJTxdHLyBTwZ1Rsa7LuGfBgOdd2SPOXU9kUDgAU39OKNB7kU0dSKgMSSp70Q5pHF6P4lbKQ3T91TXHZdfyeNeL8TqBdWn2vCTGusEpvhS7gRB1Xf6VyS7TelB6nb6fViGTWEwI2eoBHo1t0/QaYYPX9d3oJ6f2hTtAvW2q1AzTIN25fqt326XmuM6snNaLgvJpTkDpvFIbVd1+2rL6vF9S3j2M84xKy6WFp1eC+eGcQxC3hLTP1SYT6p6oVSt3rKiAf1MnzdFYYlmFaZZR8/19Wt07I0s4bQyUBTyBioOfanENHNG+zcUniAdF3cVkvuUbMiifKBr2jNHe+YqBagRezD5jXEzN2x1Lm9uzXDhRJbIH4we9f1gdT9JFKZ+S8StXG5q3fAhOdzYIfbkQ0uOUHV5XBdNibHZcWViP9ltv4M1Zky93tY7NtkuVl8zs/b2v5hab9tnar022jQC3/Kjp2vXBlLZ1BiCGYP/MLxVW0gEVm1Uh7ktVoO2hYk/on8kYJe5LmQ+jSBwaeD7gjKXJ9RPfAEsdFjEWgHbHmUxkAGFXdLa+jEj2DaioK/E+qXMsVKuv212D+2KTOQJzz0a5wA0YFlCM4a12XEdwaM85lg9urd0U5aI5V94vpphOlbQP27FtCF2G7uFsTQdswLbGZaDliWmdc1lW2HupLntyzlbkaqN1wjeGJs9wsqoe448lwyDrTD/Z2SFf/TsjOb5fYyR4+eCxT71vQCLI0s5ZSIFKtIMWJzGiNXfB+Y31HLvxxhNG2Akm7zHP7RLOm+GCN4DU2d22xiGbl2+7yYXEEZBklEMuMAUSDOaudyjaRJHQeb5nsizfW4ODeEyJpxpk3edalg9sF1K00ff1l/kcjM0DI/sPi9Mt/8c289smLDtxnZO7Jbyu80Lm0ba1eXZt40lDwluI9C9olv2q1vh/bqpCseIkHiOi/2+sMWanOCgYOywcIzAFCuGYhwUvp1o+F6auT7nNI+yiAYec2gKAlUZhQJEHLoojh1BjorNj4jSFQH2OxZQwVxMWC4SmqQCy0IaiSQIuesGziNEOapwXSrPOwayRqNhrbFUIWnSDlZSrGyPt9pcszzZg9Oy/ADQIZ4w5qnrM5f6URrTIBE48cV+TH2Bo0mW5lns8u/A03vQAzqRelnYUoKpuR4Vt2n3EBKbzN8q0pDnYRZBQp3UxeE1wsrJQmA09WzEPJ9B7D3kc29rrNP18o9rdZjP/z+97vbdnXsdW9G2utsQ/NVfgmzL/to1qG2gdg0HHaPqwg5JzbJrHzjK43W2YcVmEoVPoLhsr3h7557xmMxCkXghdQA8jB6PKfOTgEKSJCwVqZPH4Y5E1r7bYZZ8lFpaE6y7draDLHrOG6Xs1dW6BG1+ld2w2U0spPVE4xWMHF9cnb+5vHg3//lwx83DX/a1tCdHT5449zMwZTzCtgw0CoPEdmwPxeS51AVIHQTopYG7xXDBedMO5niDKrs2+z1ALvFtvTnMYplfnY1RfGh3rTf9+RhQEdZUF7igHgDWOe6grHIXO3YUorqiKM7dYAvK2iEKr+dVA98Ghpwxg0LQfWLt/piAsamaEvBWRwyK/z5yctnZegh3v2U/6p9a2BNnD/I89L089iOaB+1tOHZpFuCskgkH+33gQABsp89s7OKV8lv7zNBgwNMgTfzAGvRpkLl4acwjl3pZJFiYJVEK4SOK8a47u24+dkbdGUxTSCFKU+piz8POh7mRhXlCo8iHNMPqKzJ4pJs7w+kPdO2hm56T8SQKsUGHDCuKH8XIZhBTnmYcVZn4ruf/SKvu3WwZPm/KDNR3uZl5noOsYZVwGU4zGZZAxlFfeQY+3gCy0BfuI9zcdWfr5k4B+C5H/SRyAoCYYjnAQgCJwJ80UoeKIEjDHBKeeo8ZfcYO7braa/bePaNusKDs8/SbVZ1A5jlOGNCE5Rz5FowyFqY0CHMvdHPfCeLHwtg6uzM87JSiHxwf7vXO/80A8eXLfwBlxkQC"
expected_tables = ("FactStayNight", "DimDate", "DimProperty")
expected_paths = {
    "definition.pbism", "definition/database.tmdl", "definition/model.tmdl",
    "definition/expressions.tmdl", "definition/relationships.tmdl",
    *(f"definition/tables/{table}.tmdl" for table in expected_tables),
}
source_placeholder = "https://onelake.dfs.fabric.microsoft.com/YOUR_WORKSPACE_ID/YOUR_WAREHOUSE_ID"


def validate_definition(files, source_url):
    if set(files) != expected_paths:
        raise ValueError(f"Unexpected definition parts: {sorted(files)}")
    json.loads(files["definition.pbism"])
    text = "\n".join(files.values())
    inventory = {label: len(re.findall(rf"(?m)^[\t ]*{kind} ", text)) for label, kind in
                 (("tables", "table"), ("columns", "column"), ("measures", "measure"), ("relationships", "relationship"))}
    if inventory != {"tables": 3, "columns": 12, "measures": 3, "relationships": 2}:
        raise ValueError(f"Model inventory mismatch: {inventory}")
    expressions = files["definition/expressions.tmdl"]
    if expressions.count(f'"{source_url}"') != 1 or "AzureStorage.DataLake(OneLakeSourceUrl" not in expressions:
        raise ValueError("The model must point to the selected Warehouse through OneLake, not a SQL endpoint.")
    for table in expected_tables:
        table_text = files[f"definition/tables/{table}.tmdl"]
        if re.findall(r"(?m)^[\t ]+mode: (\S+)[\t ]*$", table_text) != ["directLake"]:
            raise ValueError(f"Expected one Direct Lake partition for {table}.")
        if not re.search(rf"(?m)^[\t ]+entityName: {table}[\t ]*$", table_text):
            raise ValueError(f"Wrong Warehouse entity for {table}.")
        if not re.search(r"(?m)^[\t ]+schemaName: dw[\t ]*$", table_text):
            raise ValueError(f"Wrong Warehouse schema for {table}.")
        if not re.search(r"(?m)^[\t ]+expressionSource: '?DirectLake - Northstar Connectivity'?[\t ]*$", table_text):
            raise ValueError(f"Wrong source expression for {table}.")
    if "sourceColumn: FullDate" not in files["definition/tables/DimDate.tmdl"]:
        raise ValueError("DimDate.Date must map to the Warehouse FullDate column.")
    return inventory


def load_template():
    raw = zlib.decompress(base64.b64decode(template_bundle_zlib_base64, validate=True))
    if hashlib.sha256(raw).hexdigest() != template_sha256:
        raise ValueError("Embedded model checksum mismatch. Use an unchanged notebook copy.")
    bundle = json.loads(raw)
    validate_definition(bundle["files"], source_placeholder)
    return bundle


def prepare_definition(files, workspace_id, warehouse_id):
    source_url = f"https://onelake.dfs.fabric.microsoft.com/{UUID(workspace_id)}/{UUID(warehouse_id)}"
    bound_files = dict(files)
    if bound_files["definition/expressions.tmdl"].count(source_placeholder) != 1:
        raise ValueError("Expected exactly one portable OneLake source binding.")
    bound_files["definition/expressions.tmdl"] = bound_files["definition/expressions.tmdl"].replace(source_placeholder, source_url, 1)
    validate_definition(bound_files, source_url)
    return {"parts": [{"path": path, "payload": base64.b64encode(content.encode("utf-8")).decode("ascii"),
                       "payloadType": "InlineBase64"} for path, content in sorted(bound_files.items())]}


template_files = load_template()["files"]
print("Embedded template verified: 3 Direct Lake tables, 12 columns, 3 measures, 2 relationships.")


In [ ]:
workspace_name = workspace_name.strip()
warehouse_name = warehouse_name.strip()
semantic_model_name = semantic_model_name.strip()
if not warehouse_name or not semantic_model_name:
    raise ValueError("Enter the Warehouse name and an unused semantic model name in Cell 2.")

if workspace_name:
    workspace = unique_name(list_all("/v1/workspaces"), workspace_name, "workspace")
    workspace_id = str(UUID(workspace["id"]))
else:
    workspace_id = str(UUID(notebookutils.runtime.context["currentWorkspaceId"]))
workspace = require_success(fabric_client.get(f"/v1/workspaces/{workspace_id}", timeout=120), "Read target workspace").json()
if not workspace.get("capacityId"):
    raise ValueError("The target workspace must be assigned to suitable Fabric capacity.")

items = list_all(f"/v1/workspaces/{workspace_id}/items")
warehouse = unique_name([item for item in items if item.get("type") == "Warehouse"], warehouse_name, "Warehouse")
warehouse_id = str(UUID(warehouse["id"]))
if any(item.get("type") == "SemanticModel" and item["displayName"].casefold() == semantic_model_name.casefold() for item in items):
    raise ValueError(f"Model {semantic_model_name!r} already exists. Choose an unused name; nothing was overwritten.")

definition = prepare_definition(template_files, workspace_id, warehouse_id)
source_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{warehouse_id}"
print(f"Workspace: {workspace['displayName']} ({workspace_id})")
print(f"Warehouse: {warehouse_name} ({warehouse_id})")
print(f"NEW model: {semantic_model_name}")
print("Source: Warehouse dw tables through OneLake, not a SQL endpoint. No Warehouse rows will be changed.")


In [ ]:
if model_created:
    raise RuntimeError(f"This session already created model {model_id}. Do not rerun creation.")
fresh_items = list_all(f"/v1/workspaces/{workspace_id}/items")
if any(item.get("type") == "SemanticModel" and item["displayName"].casefold() == semantic_model_name.casefold() for item in fresh_items):
    raise ValueError(f"Model {semantic_model_name!r} already exists. Creation stopped.")

try:
    response = fabric_client.post(
        f"/v1/workspaces/{workspace_id}/semanticModels",
        json={"displayName": semantic_model_name,
              "description": "Northstar student comparison: Direct Lake on OneLake against the student's Warehouse.",
              "definition": definition},
        timeout=120,
    )
    created_model = wait_for_operation(response, "Create semantic model")
    model_id = str(UUID(created_model["id"]))
except Exception as error:
    raise RuntimeError(
        f"Creation did not finish successfully for {semantic_model_name!r}. The request was not retried. "
        "Inspect the operation and workspace before rerunning; a new model may already exist. Nothing was updated or deleted."
    ) from error

model_created = True
print(f"Created: {semantic_model_name} ({model_id})")
print(f"Model: https://app.powerbi.com/groups/{workspace_id}/datasets/{model_id}/details")
print("Next: initial refresh of this newly created model only.")


In [ ]:
if not model_created or not model_id:
    raise RuntimeError("Create the new model successfully before refreshing it.")
if globals().get("initial_refresh_model_id") == model_id:
    raise RuntimeError("An initial refresh was already attempted for this model. Inspect refresh history and use the Fabric UI; do not resubmit blindly.")

initial_refresh_model_id = model_id
refresh_completed = False
refresh_path = f"/v1.0/myorg/groups/{workspace_id}/datasets/{model_id}/refreshes"
try:
    response = require_success(
        power_bi_client.post(refresh_path, json={"type": "full", "commitMode": "transactional", "retryCount": 0}, timeout=120),
        "Start initial Direct Lake refresh", (202,),
    )
    location = response.headers.get("Location", "")
    if not location:
        raise RuntimeError("Refresh was accepted without a tracking Location header.")
    refresh_id = str(UUID(urlparse(location).path.rstrip("/").rsplit("/", 1)[-1]))
    print(f"Initial refresh request: {refresh_id}")
    deadline = time.monotonic() + operation_timeout_seconds
    previous_status = None
    while time.monotonic() < deadline:
        details = require_success(power_bi_client.get(f"{refresh_path}/{refresh_id}", timeout=120), "Check initial refresh", (200, 202)).json()
        status = details.get("status")
        if status != previous_status:
            print(f"Initial refresh: {status}")
            previous_status = status
        if status == "Completed":
            refresh_completed = True
            break
        if status in {"Failed", "Disabled", "Cancelled", "Canceled"}:
            raise RuntimeError(json.dumps(details, ensure_ascii=True)[:3000])
        time.sleep(min(10, max(0, deadline - time.monotonic())))
    else:
        raise TimeoutError(f"Initial refresh is still pending: {refresh_id}. Check refresh history before another action.")
except Exception as error:
    raise RuntimeError(
        f"Model {semantic_model_name!r} WAS CREATED ({model_id}), but its initial refresh did not complete successfully. "
        "It has NOT been deleted. Inspect refresh history, cloud connection/SSO and Warehouse access. "
        "Complete a manual refresh in Fabric if needed, then run Cell 8; do not rerun setup or this refresh cell."
    ) from error

print("Initial Direct Lake refresh completed. It frames Warehouse data rather than importing a copy.")


In [ ]:
if not model_created or not model_id:
    raise RuntimeError("Create the new model first. If its initial refresh failed, complete it manually in Fabric before this check.")

persisted_definition_validated = False
try:
    saved = wait_for_operation(
        fabric_client.post(f"/v1/workspaces/{workspace_id}/semanticModels/{model_id}/getDefinition?format=TMDL", timeout=120),
        "Read saved model definition",
    )
    saved_files = {
        part["path"]: base64.b64decode(part["payload"], validate=True).decode("utf-8-sig").replace("\r\n", "\n")
        for part in saved["definition"]["parts"]
        if part["path"].endswith(".tmdl") or part["path"] == "definition.pbism"
    }
    inventory = validate_definition(saved_files, source_url)
    persisted_definition_validated = True
except Exception as error:
    raise RuntimeError(f"Model {model_id} exists, but saved-definition validation failed. It has not been deleted. Inspect the error before continuing.") from error

monthly_query = """
EVALUATE
SUMMARIZECOLUMNS (
    'DimDate'[YearMonth],
    TREATAS ( { 201 }, 'DimProperty'[PropertyCode] ),
    TREATAS ( { "2026-03", "2026-04" }, 'DimDate'[YearMonth] ),
    "Stay Nights", [Stay Nights],
    "Rooms Occupied", [Rooms Occupied],
    "Room Revenue", ROUND ( [Room Revenue], 2 )
)
ORDER BY 'DimDate'[YearMonth]
"""
whole_model_query = """
EVALUATE
ROW (
    "Stay Nights", [Stay Nights],
    "Rooms Occupied", [Rooms Occupied],
    "Room Revenue", ROUND ( [Room Revenue], 2 )
)
"""
try:
    monthly = fabric.evaluate_dax(dataset=model_id, dax_string=monthly_query, workspace=workspace_id)
    totals = fabric.evaluate_dax(dataset=model_id, dax_string=whole_model_query, workspace=workspace_id)
    if len(monthly) != 2 or len(totals) != 1:
        raise ValueError("Expected March and April comparison rows and one whole-model result. Check source population and access.")
    if globals().get("baseline_model_id") != model_id:
        baseline_monthly = monthly.copy()
        baseline_totals = totals.copy()
        baseline_model_id = model_id
        print("Baseline captured for this notebook session. Record it before changing Warehouse data.")
    print("Current Downtown Central results:")
    display(monthly)
    print("Current whole-model results:")
    display(totals)
    print("Original session baseline (not overwritten when this cell is rerun):")
    display(baseline_monthly)
    display(baseline_totals)
except Exception as error:
    raise RuntimeError(f"Model {model_id} exists and its definition was verified, but query checks failed. It has not been deleted. Check refresh history and Warehouse/semantic-model access.") from error

print(f"PASS: saved Direct Lake definition and comparison queries verified for {semantic_model_name}.")
print(f"Model: https://app.powerbi.com/groups/{workspace_id}/datasets/{model_id}/details")
print("Before the SQL exercise: turn OFF Keep your Direct Lake data up to date on this new model only.")
print("For before/after/reset results, rerun Cell 8 only. Do not rerun setup or creation. Use Fabric UI refresh during the exercise.")
